In [ ]:
from glob import glob
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from astropy.io import fits
import emcee
from scipy.optimize import minimize

plt.rcParams.update({
    'font.family': 'serif',
    'axes.labelsize': 12,
    'axes.titlesize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,
    'ytick.right': True,
    'xtick.minor.visible': True,
    'ytick.minor.visible': True,
})

# ── Constants ─────────────────────────────────────────────────────────────────
LI_WAVE = 6707.79
FE_WAVE = 6705.10
C_KMS   = 2.998e5

# ── RV lookup table (optional — stars not listed will have RV fit freely) ─────
RV_TABLE = {
    'HIP19793': 37.5,
    # add more stars here...
}

# ── Filename helpers ──────────────────────────────────────────────────────────
def extract_star_name(filename):
    """Extract HIPxxxxx from e.g. n0017_HIP19793_ts23_2022feb20_spectrum.fits"""
    match = re.search(r'(HIP\d+)', os.path.basename(filename))
    return match.group(1) if match else None

def find_wave_file(flux_file):
    """Find matching wave file by date token e.g. 2022feb20."""
    match = re.search(r'(\d{4}[a-z]{3}\d{2})', os.path.basename(flux_file))
    if match is None:
        raise ValueError(f"No date token found in: {flux_file}")
    date = match.group(1)
    candidates = glob(f'*{date}*wavearr.fits')
    if not candidates:
        raise FileNotFoundError(f"No wave file found for date '{date}'")
    if len(candidates) > 1:
        print(f"  Warning: multiple wave files for {date}, using: {candidates[0]}")
    return candidates[0]

# ── Spectral helpers ──────────────────────────────────────────────────────────
def get_order(wave_all, flux_all, target=LI_WAVE):
    if wave_all.ndim == 2:
        for i in range(wave_all.shape[0]):
            if wave_all[i].min() < target < wave_all[i].max():
                return wave_all[i], flux_all[i]
        raise ValueError("No order found covering λ6707.79 Å")
    return wave_all, flux_all

def model(lam, m, b, C1, C2, sigma, RVs):
    cont  = m * (lam - 6707.0) + b
    g_li  = C1 * np.exp(-0.5 * ((LI_WAVE - RVs - lam) / sigma)**2)
    g_ref = C2 * np.exp(-0.5 * ((FE_WAVE - RVs - lam) / sigma)**2)
    return cont - g_li - g_ref

def neg_log_like(theta, w, f_norm, f_err):
    m, b, C1, C2, sigma, RVs = theta
    if sigma <= 0 or C1 < 0 or C2 < 0:
        return np.inf
    pred = model(w, *theta)
    chi2 = np.sum(((f_norm - pred) / f_err)**2)
    return 0.5 * chi2

def log_prior(theta):
    m, b, C1, C2, sigma, RVs = theta
    if (0 < sigma < 5 and 0 <= C1 < 0.8 and 0 <= C2 < 0.8
            and -0.5 < m < 0.5 and 0.5 < b < 1.5 and -3 < RVs < 3):  # ±3 Å ≈ ±130 km/s
        return 0.0
    return -np.inf

def log_prob(theta, w, f_norm, f_err):
    lp = log_prior(theta)
    return lp - neg_log_like(theta, w, f_norm, f_err) if np.isfinite(lp) else -np.inf

def ew_mA(theta):
    m, b, C1, C2, sigma, RVs = theta
    H = m * (LI_WAVE - 6707.0) + b
    return np.sqrt(2 * np.pi) * sigma * C1 / H * 1000.0 if H > 0 else np.nan

def ul_curve(ew_ul_mA, params_ref, w_fine):
    m, b, _, C2, sigma, RVs = params_ref
    H     = m * (LI_WAVE - 6707.0) + b
    C1_ul = (ew_ul_mA / 1000.0) * H / (np.sqrt(2 * np.pi) * sigma)
    return model(w_fine, m, b, C1_ul, C2, sigma, RVs)

# ── Per-star processing ───────────────────────────────────────────────────────
def process_star(flux_file, wave_file, star_name, rv_kms=None):
    print(f"\n{'='*60}")
    print(f"Processing {star_name}")
    print(f"  Flux : {flux_file}")
    print(f"  Wave : {wave_file}")

    flux_all = fits.getdata(flux_file).astype(float)
    wave_all = fits.getdata(wave_file).astype(float)

    wave_raw, flux_raw = get_order(wave_all, flux_all)

    # If RV known, pre-correct; otherwise let model fit RVs freely
    if rv_kms is not None:
        wave_rest = wave_raw / (1.0 + rv_kms / C_KMS)
        print(f"  RV pre-corrected: {rv_kms} km/s")
    else:
        wave_rest = wave_raw
        print(f"  RV: free parameter (fit by MCMC)")

    mask = (wave_rest >= 6704.0) & (wave_rest <= 6710.0)
    w, f = wave_rest[mask], flux_raw[mask]

    cont_mask = ((w > 6704.0) & (w < 6704.8)) | ((w > 6709.2) & (w < 6710.0))
    continuum = np.median(f[cont_mask]) if cont_mask.sum() > 3 else np.percentile(f, 90)
    f_norm    = f / continuum
    noise_std = np.std(f_norm[cont_mask]) if cont_mask.sum() > 3 else 0.01
    f_err     = np.full_like(f_norm, noise_std)

    snr = 1.0 / noise_std if noise_std > 0 else 50.0
    print(f"  SNR/pixel ≈ {snr:.0f},  σ_cont ≈ {noise_std:.4f}")

    # MAP fit
    p0     = [0.0, 1.0, 0.05, 0.10, 0.5, 0.0]
    result = minimize(
        neg_log_like, p0, args=(w, f_norm, f_err),
        method='Nelder-Mead',
        options={'maxiter': 100000, 'xatol': 1e-7, 'fatol': 1e-7}
    )
    p_best = result.x
    print(f"  Best-fit: C1={p_best[2]:.4f}, C2={p_best[3]:.4f}, "
          f"σ={p_best[4]:.4f} Å, RVs={p_best[5]:.4f} Å")

    # MCMC
    ndim, nwalkers = 6, 32
    pos = p_best + 1e-4 * np.random.randn(nwalkers, ndim)
    sampler = emcee.EnsembleSampler(nwalkers, ndim, log_prob, args=(w, f_norm, f_err))
    print("  Burn-in...")
    sampler.run_mcmc(pos, 500, progress=True)
    pos2 = sampler.get_last_sample().coords
    sampler.reset()
    print("  Production...")
    sampler.run_mcmc(pos2, 2000, progress=True)
    flat = sampler.get_chain(discard=0, flat=True)

    # EW statistics
    ew_samples = np.array([ew_mA(s) for s in flat])
    ew_samples = ew_samples[np.isfinite(ew_samples)]
    ew_med     = np.median(ew_samples)
    ew_lo, ew_hi = np.percentile(ew_samples, [16, 84])
    ew_err     = 0.5 * (ew_hi - ew_lo)
    ew_1sig, ew_2sig, ew_3sig = np.percentile(ew_samples, [68, 95, 99.7])

    # Fitted RV
    RVs_samples   = flat[:, 5]
    RVs_med       = np.median(RVs_samples)
    RVs_err       = 0.5 * (np.percentile(RVs_samples, 84) - np.percentile(RVs_samples, 16))
    rv_fitted_kms = RVs_med / LI_WAVE * C_KMS

    print(f"  EW = {ew_med:.1f} ± {ew_err:.1f} mÅ")
    print(f"  Upper limits — 1σ: {ew_1sig:.1f}  2σ: {ew_2sig:.1f}  3σ: {ew_3sig:.1f} mÅ")
    print(f"  Fitted RV shift: {RVs_med:.4f} Å ({rv_fitted_kms:.1f} km/s)")

    # Plot
    LIGHT_BLUE  = '#a8c8f0'
    DARK_BLUE   = '#1a4f8a'
    w_fine      = np.linspace(6704.0, 6710.0, 1000)
    f_bestfit   = model(w_fine, *p_best)
    idx_draws   = np.random.choice(len(flat), 200, replace=False)
    post_curves = [model(w_fine, *flat[i]) for i in idx_draws]
    p_med       = np.median(flat, axis=0)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left panel — posterior fit
    ax = axes[0]
    ax.errorbar(w, f_norm, yerr=f_err, fmt='k+', ms=5, lw=0.8,
                capsize=0, zorder=3, label='Observed')
    for curve in post_curves:
        ax.plot(w_fine, curve, color=LIGHT_BLUE, alpha=0.06, lw=0.7, zorder=1)
    ax.plot(w_fine, f_bestfit, color=DARK_BLUE, lw=1.8, zorder=4, label='Fit')
    ax.set_xlim(6704, 6710)
    ax.set_xlabel('Wavelength (Å)')
    ax.set_ylabel('Flux')
    ax.legend(loc='lower right', fontsize=9, framealpha=0.8)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.xaxis.set_minor_locator(ticker.MultipleLocator(0.2))
    ax.text(0.05, 0.05,
            f'{star_name}\nEW$_{{\\rm Li}}$ = {ew_med:.0f} ± {ew_err:.0f} mÅ',
            transform=ax.transAxes, fontsize=10,
            va='bottom', bbox=dict(boxstyle='round', fc='white', alpha=0.7))

    # Right panel — upper limits
    ax2 = axes[1]
    ax2.errorbar(w, f_norm, yerr=f_err, fmt='k+', ms=5, lw=0.8,
                 capsize=0, zorder=3, label='Observed')
    ax2.plot(w_fine, ul_curve(ew_1sig, p_med, w_fine),
             color='goldenrod', lw=1.8, label=f'1σ UL = {ew_1sig:.0f} mÅ')
    ax2.plot(w_fine, ul_curve(ew_2sig, p_med, w_fine),
             color='crimson',   lw=1.8, label=f'2σ UL = {ew_2sig:.0f} mÅ')
    ax2.plot(w_fine, ul_curve(ew_3sig, p_med, w_fine),
             color=DARK_BLUE,   lw=1.8, label=f'3σ UL = {ew_3sig:.0f} mÅ')
    ax2.set_xlim(6704, 6710)
    ax2.set_xlabel('Wavelength (Å)')
    ax2.set_ylabel('Flux')
    ax2.legend(loc='lower right', fontsize=9, framealpha=0.8)
    ax2.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax2.xaxis.set_minor_locator(ticker.MultipleLocator(0.2))

    plt.suptitle(f'Li λ6707.79 — {star_name}', y=1.01, fontsize=12)
    plt.tight_layout()
    out_png = f'lithium_{star_name}.png'
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {out_png}")

    return {
        'star':       star_name,
        'ew_med':     ew_med,
        'ew_err':     ew_err,
        'ew_1sig':    ew_1sig,
        'ew_2sig':    ew_2sig,
        'ew_3sig':    ew_3sig,
        'rv_kms':     rv_kms if rv_kms is not None else rv_fitted_kms,
        'rv_source':  'table' if rv_kms is not None else 'fit',
    }

# ── Main loop ─────────────────────────────────────────────────────────────────
results = []
for flux_file in sorted(glob('*_spectrum.fits')):
    star_name = extract_star_name(flux_file)
    if star_name is None:
        print(f"Skipping (no HIP name): {flux_file}")
        continue

    rv_kms = RV_TABLE.get(star_name)  # None if not in table — RV fit freely

    try:
        wave_file = find_wave_file(flux_file)
        res = process_star(flux_file, wave_file, star_name, rv_kms)
        results.append(res)
    except Exception as e:
        print(f"  ERROR on {star_name}: {e}")

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'='*72}")
print(f"{'Star':<12} {'RV (km/s)':<18} {'EW (mÅ)':<16} {'1σ UL':<10} {'2σ UL':<10} {'3σ UL':<10}")
print('-' * 72)
for r in results:
    rv_str = f"{r['rv_kms']:.1f} ({r['rv_source']})"
    print(f"{r['star']:<12} {rv_str:<18} {r['ew_med']:.1f} ± {r['ew_err']:.1f}{'':>4}"
          f"{r['ew_1sig']:<10.1f} {r['ew_2sig']:<10.1f} {r['ew_3sig']:<10.1f}")
print(f"{'='*72}")
print(f"Total processed: {len(results)} stars")


Processing HIP54028
  Flux : n0007_HIP54028_ts23_2024may19_spectrum.fits
  Wave : ts23_2024may19_thar_solution_wavearr.fits
  RV: free parameter (fit by MCMC)
  SNR/pixel ≈ 10,  σ_cont ≈ 0.0991
  Best-fit: C1=0.0000, C2=0.0000, σ=2.1857 Å, RVs=-0.0127 Å
  Burn-in...


  0%|                                                                                          | 0/500 [00:00<?, ?it/s]C:\Users\ejo08\miniforge3\envs\baffles\lib\site-packages\emcee\moves\red_blue.py:99: RuntimeWarning: invalid value encountered in scalar subtract
  lnpdiff = f + nlp - state.log_prob[j]
100%|███████████████████████████████████████████████████████████████████████████████| 500/500 [00:01<00:00, 404.28it/s]


  Production...


100%|█████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:04<00:00, 428.20it/s]


  EW = 266.9 ± 508.1 mÅ
  Upper limits — 1σ: 524.2  2σ: 2478.4  3σ: 4415.2 mÅ
  Fitted RV shift: 2.0142 Å (90.0 km/s)
  Saved: lithium_HIP54028.png

Processing HIP47741
  Flux : n0008_HIP47741_ts23_2024may19_spectrum.fits
  Wave : ts23_2024may19_thar_solution_wavearr.fits
  RV: free parameter (fit by MCMC)
  SNR/pixel ≈ 7,  σ_cont ≈ 0.1442
  Best-fit: C1=0.0462, C2=0.0000, σ=1.0393 Å, RVs=-0.0028 Å
  Burn-in...


...

100%|█████████████████████████████████████████████████████████████████████████████| 2000/2000 [00:06<00:00, 300.48it/s]


  EW = 381.9 ± 607.1 mÅ
  Upper limits — 1σ: 712.4  2σ: 2132.8  3σ: 3438.3 mÅ
  Fitted RV shift: -1.8071 Å (-80.8 km/s)
  Saved: lithium_HIP89583.png
Skipping (no HIP name): n0171_ROXs42B_ts23_2022feb21_spectrum.fits

Processing HIP102807
  Flux : n0172_HIP102807_ts23_2022jul20_spectrum.fits
  Wave : ts23_2022jul20_thar_solution_wavearr.fits
  RV: free parameter (fit by MCMC)
  SNR/pixel ≈ 41,  σ_cont ≈ 0.0244
  Best-fit: C1=0.0018, C2=0.0000, σ=1.4303 Å, RVs=-0.0024 Å
  Burn-in...


100%|███████████████████████████████████████████████████████████████████████████████| 500/500 [00:01<00:00, 259.58it/s]


  Production...


 40%|███████████████████████████████▏                                              | 799/2000 [00:03<00:04, 266.70it/s]